# 06 JPS：网格对称剪枝


本课按百科条目写。Jump Point Search 只加速 **均匀代价的规则网格**，拓扑路网不要用。调度台在最后。

---

## 1. 定义

**Jump Point Search**（Harabor & Grastien, AAAI 2011）在四连通或八连通均匀网格上跑 A\*，但后继不是「四个邻居」，而是沿直线 **跳** 到下一个有决策价值的格子（跳点）。

跳点：沿方向 d 走，直到

- 碰到终点，或
- **被迫邻居**（forced neighbour）：旁边有墙，使得不在此处转弯就会错过某条最短路，或
- 本课教学策略：这个方向走不下去时，返回 **最后一格可走格子**（保证四连通完备；论文原版在无 forced 时返回空，再靠自然邻居递归）。

输入：`GridMap`、起终点。输出：与 A\* 相同代价的格子路径（中间被跳过的格子要填回去）。

---

## 2. 为何能剪、何时不能用

空地从 (0,0) 到 (3,2)：先右后下、先下后右，边数相同。A\* 会把中间所有排列都扩一遍。JPS 向右一直跳，直到该方向出现墙角或终点。

**只适用于规则网格。** 任意拓扑没有「沿轴跳」的几何，本仓库对 `TopoMap` 返回 `inapplicable`。边权不均为 1 时，跳过的格子代价对不上，也不要用。

八连通 JPS 的 forced 规则与四连通不同（对角移动会「擦」墙角）。本课只写 **四连通**。

---

## 3. 不变量

- 跳点之间是直线、中间无墙。
- 填回直线格子后，路径合法，边数 = A\* 最优边数（教学版返回最后一格，完备）。
- 扩展次数 ≤ A\*（空地、长走廊上少很多；迷宫墙多时优势变小）。

---

## 4. 伪代码

```
function JPS(grid, s, t):
    同 A*，但 Successors(u) =
        for 方向 d in {右,左,下,上}:
            jp ← Jump(u, d)
            if jp ≠ nil 且 jp ≠ u: 产生 jp，代价 = 曼哈顿(u, jp)

Jump(x, d):
    n ← 沿 d 走一步
    若出界或墙: return nil（或教学：没有下一步就停在上一格由调用方处理）
    若 n = t 或 Forced(n, d): return n
    further ← Jump(n, d)
    若 further ≠ nil: return further
    return n                 # 教学：此方向最后一格

Forced(n, d) 对水平 d=(±1,0):
    上侧是墙 且 右上/左上（沿 d）可走，或下侧对称
竖直方向对换 x/y。
```

主循环结束后：`came_from` 只记跳点。输出前在相邻跳点间按方向逐步插入中间格。

---

## 5. 手算 forced

```
. . .
# n .
. . .
```

在 n 向右走（d=+x）。n 的 **上方是墙**，右上若可走，则「只继续向右」会错过经右上的路 → n 是跳点，必须在此把右上当成后继（四连通没有斜向，forced 仍标记「这里要停，让 A\* 换方向」）。

空 8×8、无墙、S 左上 G 右下：向右跳会一口气跳到右边界附近，扩展远小于 A\*。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：forced / jump / 填格子

在 8×8 空网格上自己写。邻居顺序仍是右左下上。


In [ ]:
import heapq
from lib.maps import grid_open, grid_wall, topo_oneway
from lib.pretty import draw_grid

grid = grid_open(8, 8)
START, GOAL = (0, 0), (7, 7)


def blocked(x, y):
    return not grid.passable((x, y))


def forced(nx, ny, dx, dy):
    # 墙在侧面、沿前进方向的「斜侧」可走 ⇒ 必须在此转向
    if dx != 0 and dy == 0:
        return (blocked(nx, ny - 1) and not blocked(nx + dx, ny - 1)) or (
            blocked(nx, ny + 1) and not blocked(nx + dx, ny + 1)
        )
    if dy != 0 and dx == 0:
        return (blocked(nx - 1, ny) and not blocked(nx - 1, ny + dy)) or (
            blocked(nx + 1, ny) and not blocked(nx + 1, ny + dy)
        )
    return False


def jump(x, y, dx, dy):
    nx, ny = x + dx, y + dy
    if blocked(nx, ny):
        return None
    if (nx, ny) == GOAL or forced(nx, ny, dx, dy):
        return (nx, ny)
    further = jump(nx, ny, dx, dy)
    if further is not None:
        return further
    return (nx, ny)


def successors(node):
    x, y = node
    succs = []
    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        jp = jump(x, y, dx, dy)
        if jp and jp != node:
            succs.append(jp)
    return succs


print("S 的跳点（空地应是沿四轴跳到边界前）:", successors(START))
print("中点 (3,3) 的跳点:", successors((3, 3)))


空地上 `jump` 会走到该方向最后一格（教学完备策略），所以 S 向右的跳点是 (7,0)，向下是 (0,7)。不是四邻居。

## 7. 主循环 = A*，后继换成跳点


In [ ]:
def reconstruct_jumps(came, goal):
    node = goal
    path = [node]
    while came[node] is not None:
        node = came[node]
        path.append(node)
    path.reverse()
    return path


def fill_straight(path):
    # 跳点之间可能隔了好几格。按轴逐步插入，否则路径「瞬移」。
    filled = [path[0]]
    for a, b in zip(path, path[1:]):
        dx = 0 if b[0] == a[0] else (1 if b[0] > a[0] else -1)
        dy = 0 if b[1] == a[1] else (1 if b[1] > a[1] else -1)
        p = a
        while p != b:
            p = (p[0] + dx, p[1] + dy)
            filled.append(p)
    return filled


def jps_search(start, goal, verbose=True):
    heap = []
    g = {start: 0.0}
    came = {start: None}
    heapq.heappush(heap, (grid.heuristic(start, goal), start))
    closed = set()
    expanded = 0
    while heap:
        _f, cur = heapq.heappop(heap)
        if cur in closed:
            continue
        closed.add(cur)
        expanded += 1
        if verbose and expanded <= 12:
            print(f"扩展跳点 {cur}  g={g[cur]:.0f}  后继={successors(cur)}")
        if cur == goal:
            jumps = reconstruct_jumps(came, goal)
            filled = fill_straight(jumps)
            return filled, float(len(filled) - 1), expanded, jumps
        for jp in successors(cur):
            step = abs(jp[0] - cur[0]) + abs(jp[1] - cur[1])
            ng = g[cur] + step
            if ng < g.get(jp, float("inf")):
                g[jp] = ng
                came[jp] = cur
                heapq.heappush(heap, (ng + grid.heuristic(jp, goal), jp))
    return None, float("inf"), expanded, []


path, cost, n_exp, jumps = jps_search(START, GOAL)
print("跳点序列", jumps)
print("填格后边数", cost, "扩展", n_exp)
print(draw_grid(8, 8, start=START, goal=GOAL, path=path, closed=jumps))


| 行 | 作用 |
|----|------|
| `successors` | 四方向各 jump 一次，不是四邻居 |
| `step = 曼哈顿(cur,jp)` | 中间直线代价=格数 |
| `fill_straight` | 给执行层连续格子 |
| 关闭集合 | 仍是跳点，不是每一个中间格 |

空 8×8 最优边数 14。JPS 扩展应明显小于 A\*（A\* 大约几十，JPS 往往十几或更少）。


In [ ]:
from lib.search import astar
from lib.algos.single_agent import jps as jps_lib

g_open = grid_open(8, 8)
a = astar(g_open, START, GOAL, record=False)
print("空地 A* 扩展", a.expanded, "代价", a.cost, "本课 JPS 扩展", n_exp, "代价", cost)

gw = grid_wall()
aw = astar(gw, START, GOAL, record=False)
jw = jps_lib(gw, START, GOAL, record=False)
print("墙图 A*", aw.expanded, aw.cost, "库JPS", jw.expanded, jw.cost)
print("拓扑应 inapplicable:", jps_lib(topo_oneway(), "I0", "G1").extra)


## 常见 bug

1. 忘了 `fill_straight`：路径从跳点瞬移，执行器会穿墙或一步走多格。
2. 八连通 forced 抄到四连通（或反过来）。
3. 把 JPS 用在路口拓扑、不等权边。
4. `jump` 前方无跳点就返回 None，四连通可能不完备；本课返回最后一格。
5. 比较扩展次数时用了 `record=True` 的 A\* 和不同地图。

## 条目小结

| 项目 | 内容 |
|------|------|
| 适用 | 均匀规则网格 |
| 核心 | 直线跳过对称路径 |
| 保证 | 教学版完备，代价=A\* |
| 不适用 | 拓扑、不等权、任意多边形 |
| 下一课 | 多车之前先把「谁在何时占哪」锁清楚 |

## 练习

1. 空 8×8，S=(0,0)，G=(7,0) 同一行。JPS 第一次向右跳到哪？扩展几次？
2. 在 (3,1) 放一堵墙，(3,0) 向右走时 forced 会不会为真？画上下侧。
3. 证明：跳点间填直线后，边数等于曼哈顿（无墙）或不少于曼哈顿（有墙）。


In [ ]:
from lib.studio.widget import PlannerStudio
from lib.algos.single_agent import jps as jps_lib

r = jps_lib(gw, START, GOAL)
studio = PlannerStudio()
studio.show_search({**gw.to_studio(), "start": [0, 0], "goal": [7, 7]}, r, "JPS")
studio
